## Tool Calling<br>
Agent 将所有工具的描述与参数定义随请求发给model，model自行判断是否调用、调用哪个工具并输出调用指令，再由 Agent 解析指令执行对应工具并将结果回传给model循环处理直至得出最终回答。

In [19]:
# model
from rich import print as rprint
from dotenv import load_dotenv
load_dotenv(override=True)
from langchain.chat_models import init_chat_model
model = init_chat_model(
    model="deepseek-v4-flash", # 模型名称
)

### 1、定义工具

In [21]:
from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    """
    获取城市的天气信息
    :param city: 城市名称，例如 "上海", "北京"
    :return: 天气信息字符串
    """
    return city + "晴天，温度15°C"


In [6]:
# 测试
get_weather.invoke({"city": "苏州"})

'苏州晴天，温度15°C'

### 2、model调用工具

In [23]:
# model绑定工具
model_with_tools = model.bind_tools([get_weather])

# model自行决定是否调用工具
response = model_with_tools.invoke("苏州的天气如何？")
rprint(response)

AIMessage(
    content='',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': '用户想知道苏州的天气。让我调用get_weather工具来获取苏州的天气信息。'
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 64,
            'prompt_tokens': 299,
            'total_tokens': 363,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 19,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256},
            'prompt_cache_hit_tokens': 256,
            'prompt_cache_miss_tokens': 43
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-v4-flash',
        'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
        'id': '3b1a02d9-11a0-4a50-b107-0ed2a2eda1ab',
        'finish_reason': 'tool_calls',
        'logprobs': None
    },
    id='lc_run--019f70b1-442b-7091-aefa-05ae21ae6492-0',
    tool_calls=[
        {
            'name': 'get_weather',
            'args': {'city': '苏州'},
            'id': 'call_00_JfvIH9QLkrmFFkS4aNYx0584',
            'type': 'tool_call'
        }
    ],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 299,
        'output_tokens': 64,
        'total_tokens': 363,
        'input_token_details': {'cache_read': 256},
        'output_token_details': {'reasoning': 19}
    }
)

![tool](../img/tool.png)